[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/experiments/summarization.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">摘要服务的提示词模板迭代</h1>

假设你正在为媒体公司部署摘要模型服务，把每日新闻浓缩为在线展示的简短摘要。使用 LLM 做摘要的一项挑战是，即使最好的模型也往往过于冗长。

本教程将构建数据集并运行实验，设计能生成简洁且准确摘要的提示词模板。你将：

- 向 Phoenix 上传包含文章与人工参考摘要的**示例数据集**
- 定义为新闻文章生成摘要的**实验任务**
- 设计衡量长度和 ROUGE 分数的**评估器**
- 运行**实验**迭代提示词模板，并比较不同 LLM 生成的摘要

⚠️ 本教程需要 OpenAI API 密钥，可选使用 Anthropic API 密钥。

开始吧！


## 安装依赖并导入库

安装所需依赖并导入库。

In [ ]:
%pip install -U arize-phoenix anthropic openai openinference-instrumentation-openai openinference-instrumentation-anthropic rouge tiktoken datasets pandas

In [ ]:
from typing import Any, Dict

import pandas as pd

pd.set_option("display.max_colwidth", None)  # 完整显示 DataFrame 单元内容

## 启动 Phoenix

启动 Phoenix，按照单元输出中的说明打开 Phoenix UI。

In [ ]:
import phoenix as px

px.launch_app().view()

In [ ]:
from phoenix.client import AsyncClient

px_client = AsyncClient()

## 为应用插桩


In [ ]:
from openinference.instrumentation.anthropic import AnthropicInstrumentor
from openinference.instrumentation.openai import OpenAIInstrumentor
from phoenix.otel import register

tracer_provider = register(endpoint="http://127.0.0.1:6006/v1/traces")
OpenAIInstrumentor().instrument(skip_dep_check=True, tracer_provider=tracer_provider)
AnthropicInstrumentor().instrument(skip_dep_check=True, tracer_provider=tracer_provider)

## 创建数据集

从 HuggingFace 下载[数据](https://huggingface.co/datasets/abisee/cnn_dailymail)，随机查看十行。数据集包含新闻文章和人工摘要，我们将以人工摘要为参考，与 LLM 生成的摘要比较。

将数据作为 Phoenix **数据集**上传，点击单元输出中的链接查看各个**示例**。稍后将在此数据集上运行**实验**，迭代改进摘要应用。

In [ ]:
from datetime import datetime, timezone

from datasets import load_dataset

hf_ds = load_dataset("abisee/cnn_dailymail", "3.0.0")
df = (
    hf_ds["test"]
    .to_pandas()
    .sample(n=10, random_state=0)
    .reset_index(drop=True)
    .rename(columns={"highlights": "summary"})
)
dataset = await px_client.datasets.create_dataset(
    name=f"news-article-summaries-{datetime.now(timezone.utc)}",
    dataframe=df,
    input_keys=["article"],
    output_keys=["summary"],
)

## 定义实验任务

**任务**是一个可调用对象，通过调用链、查询引擎或 LLM，将数据集示例的输入映射为输出。**实验**对数据集中的所有示例执行任务，并可选运行**评估器**为任务输出评分。

先定义任务，本例调用 OpenAI。如果环境变量中尚未设置 OpenAI API 密钥，先进行设置。

In [ ]:
import os
from getpass import getpass

if os.environ.get("OPENAI_API_KEY") is None:
    os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥：")

接下来定义函数，格式化提示词模板，并对一个示例调用 OpenAI 模型。

In [ ]:
from openai import AsyncOpenAI

openai_client = AsyncOpenAI()


async def summarize_article_openai(input: Any, prompt_template: str, model: str) -> str:
    formatted_prompt_template = prompt_template.format(article=input["article"])
    response = await openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "assistant", "content": formatted_prompt_template},
        ],
    )
    assert response.choices
    return response.choices[0].message.content

使用 `functools.partial` 从此函数派生第一个任务，即接收示例并返回输出的可调用对象。在测试示例上调用，试一下任务。

In [ ]:
import textwrap
from functools import partial

template = """
总结文章：

ARTICLE
=======
{article}

SUMMARY
=======
"""
gpt_4o = "gpt-4o-2024-05-13"
task = partial(summarize_article_openai, prompt_template=template, model=gpt_4o)
test_example = dataset.examples[0]
print(textwrap.fill(await task(test_example["input"]), width=100))

## 定义评估器

评估器接收任务输出（本例为字符串）并评分，通常借助 LLM。这里创建 ROUGE 评分评估器，将 LLM 摘要与作为数据集一部分上传的人工参考摘要比较。ROUGE 有多个变体，为简单起见使用 ROUGE-1：

- ROUGE-1 精确率：生成摘要中与参考摘要重叠的 token 比例，即重叠 token 数 / 生成摘要 token 数。
- ROUGE-1 召回率：参考摘要中与生成摘要重叠的 token 比例，即重叠 token 数 / 参考摘要 token 数。
- ROUGE-1 F1 分数：精确率和召回率的调和平均数，用一个数字平衡二者。

ROUGE 分数越高，生成摘要与参考摘要越相似。接近 1 / 2 的分数已非常优秀；[在此数据集上微调的模型取得了约 0.44 的 ROUGE 分数](https://huggingface.co/datasets/abisee/cnn_dailymail#supported-tasks-and-leaderboards)。

由于也关注简洁性，还将定义评估器统计每条生成摘要的 token 数。

定义评估器时可使用任何第三方库，本例使用 `rouge` 和 `tiktoken`。

In [ ]:
import tiktoken
from rouge import Rouge


# 便捷函数
def _rouge_1(hypothesis: str, reference: str) -> Dict[str, Any]:
    scores = Rouge().get_scores(hypothesis, reference)
    return scores[0]["rouge-1"]


def _rouge_1_f1_score(hypothesis: str, reference: str) -> float:
    return _rouge_1(hypothesis, reference)["f"]


def _rouge_1_precision(hypothesis: str, reference: str) -> float:
    return _rouge_1(hypothesis, reference)["p"]


def _rouge_1_recall(hypothesis: str, reference: str) -> float:
    return _rouge_1(hypothesis, reference)["r"]


# 评估器
def rouge_1_f1_score(output: str, expected: Dict[str, Any]) -> float:
    return _rouge_1_f1_score(hypothesis=output, reference=expected["summary"])


def rouge_1_precision(output: str, expected: Dict[str, Any]) -> float:
    return _rouge_1_precision(hypothesis=output, reference=expected["summary"])


def rouge_1_recall(output: str, expected: Dict[str, Any]) -> float:
    return _rouge_1_recall(hypothesis=output, reference=expected["summary"])


def num_tokens(output: str) -> int:
    encoding = tiktoken.encoding_for_model(gpt_4o)
    return len(encoding.encode(output))


EVALUATORS = [rouge_1_f1_score, rouge_1_precision, rouge_1_recall, num_tokens]

## 运行实验并迭代提示词模板


运行第一个实验，点击单元输出链接，检查任务输出（生成摘要）和评估。

In [ ]:
initial_experiment_results = await px_client.experiments.run_experiment(
    dataset=dataset,
    task=task,
    experiment_name="initial-template",
    experiment_description="使用简单提示词模板的第一个实验",
    experiment_metadata={"vendor": "openai", "model": gpt_4o},
    evaluators=EVALUATORS,
)

In [ ]:
def show_evaluation_summary(exp):
    rouge_f1_scores = [
        run.result["score"]
        for run in exp["evaluation_runs"]
        if run.result and "score" in run.result and run.name == "rouge_1_f1_score"
    ]
    rouge_precision_scores = [
        run.result["score"]
        for run in exp["evaluation_runs"]
        if run.result and "score" in run.result and run.name == "rouge_1_precision"
    ]
    rouge_recall_scores = [
        run.result["score"]
        for run in exp["evaluation_runs"]
        if run.result and "score" in run.result and run.name == "rouge_1_recall"
    ]
    token_counts = [
        run.result["score"]
        for run in exp["evaluation_runs"]
        if run.result and "score" in run.result and run.name == "num_tokens"
    ]

    print("Template results:")
    print(
        f"  ROUGE-1 F1:        {sum(rouge_f1_scores) / len(rouge_f1_scores):.3f} (n={len(rouge_f1_scores)})"
    )
    print(
        f"  ROUGE-1 精确率： {sum(rouge_precision_scores) / len(rouge_precision_scores):.3f} (n={len(rouge_precision_scores)})"
    )
    print(
        f"  ROUGE-1 召回率：    {sum(rouge_recall_scores) / len(rouge_recall_scores):.3f} (n={len(rouge_recall_scores)})"
    )
    print(
        f"  平均 token 数：    {sum(token_counts) / len(token_counts):.1f} (n={len(token_counts)})"
    )


show_evaluation_summary(initial_experiment_results)

初始提示词模板的指导很少，ROUGE-1 F1 分数略高于 0.3，具体值每次运行会变化。检查任务输出，也会发现生成摘要远比参考摘要冗长，因此 ROUGE-1 召回率高、精确率低。尝试改进提示词，让摘要更简洁，平衡召回率和精确率，同时保持或提高 F1。先明确要求 LLM 生成简洁摘要。

In [ ]:
concise_template = """
用两到四句话总结文章。保持简洁，仅包含最重要信息。

文章
=======
{article}

摘要
=======
"""
task = partial(summarize_article_openai, prompt_template=concise_template, model=gpt_4o)
second_experiment_results = await px_client.experiments.run_experiment(
    dataset=dataset,
    task=task,
    experiment_name="concise-template",
    experiment_description="明确要求 LLM 保持简洁",
    experiment_metadata={"vendor": "openai", "model": gpt_4o},
    evaluators=EVALUATORS,
)

show_evaluation_summary(second_experiment_results)

检查实验结果，会发现平均 `num_tokens` 确实增加了，但生成摘要仍远比参考摘要冗长。

不要只要求 LLM 生成简洁摘要，试试用少样本提示词展示文章和优秀摘要。下面的单元在更新后的模板中加入了几篇文章及参考摘要。

In [ ]:
train_df = (
    hf_ds["train"]
    .to_pandas()
    .sample(n=5, random_state=42)
    .head()
    .rename(columns={"highlights": "summary"})
)

example_template = """
文章
=======
{article}

摘要
=======
{summary}
"""

examples = "\n".join(
    [
        example_template.format(article=row["article"], summary=row["summary"])
        for _, row in train_df.iterrows()
    ]
)

few_shot_template = """
用两到四句话总结文章。保持简洁，仅包含最重要信息，参考以下示例。

示例
========

{examples}


现在总结以下文章。

文章
=======
{article}

摘要
=======
"""

few_shot_template = few_shot_template.format(
    examples=examples,
    article="{article}",
)
print(few_shot_template)

现在运行实验。

In [ ]:
task = partial(summarize_article_openai, prompt_template=few_shot_template, model=gpt_4o)
few_shot_experiment_results = await px_client.experiments.run_experiment(
    dataset=dataset,
    task=task,
    experiment_name="few-shot-template",
    experiment_description="include examples",
    experiment_metadata={"vendor": "openai", "model": gpt_4o},
    evaluators=EVALUATORS,
)

show_evaluation_summary(few_shot_experiment_results)

加入示例后，每条摘要的 token 数会大幅下降，同时保持 F1。

## 与另一个模型比较（可选）

⚠️ 本节需要 Anthropic API 密钥。

现在模板表现已比较理想，可以比较其他模型在此任务上的表现。Anthropic 的 Claude 以简洁、直截了当的输出著称。

如果尚未设置，先输入 Anthropic API 密钥。


In [ ]:
import os
from getpass import getpass

if os.environ.get("ANTHROPIC_API_KEY") is None:
    os.environ["ANTHROPIC_API_KEY"] = getpass("🔑 请输入 Anthropic API 密钥：")

接下来定义新任务，用相同提示词模板总结文章，然后运行实验。任务可选接受特殊参数名，它们会绑定到特定值。`example` 绑定到实验运行关联的数据集示例。

In [ ]:
from anthropic import AsyncAnthropic

anthropic_client = AsyncAnthropic()


async def summarize_article_anthropic(input: Any, prompt_template: str, model: str) -> str:
    formatted_prompt_template = prompt_template.format(article=input["article"])
    message = await anthropic_client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[{"role": "user", "content": formatted_prompt_template}],
    )
    return message.content[0].text


claude_35_sonnet = "claude-3-5-sonnet-20240620"
task = partial(
    summarize_article_anthropic, prompt_template=few_shot_template, model=claude_35_sonnet
)

anthropic_experiment_results = await px_client.experiments.run_experiment(
    dataset=dataset,
    task=task,
    experiment_name="anthropic-few-shot",
    experiment_description="anthropic",
    experiment_metadata={"vendor": "anthropic", "model": claude_35_sonnet},
    evaluators=EVALUATORS,
)

show_evaluation_summary(anthropic_experiment_results)

如果实验没有生成更简洁的摘要，检查各条结果。你可能发现 Claude 3.5 Sonnet 的部分摘要以下面的开场白开始：

```
Here is a concise 3-sentence summary of the article...
```

尝试调整提示词并重新运行实验，让 Claude 输出不包含这段开场白。这样应能得到迄今最简洁的摘要。

## 回顾与后续步骤

恭喜！本教程中你已经：

- 创建 Phoenix 数据集
- 定义实验任务和自定义评估器
- 迭代改进提示词模板，生成更简洁的摘要，并平衡 ROUGE-1 精确率和召回率

接下来可继续迭代提示词模板。如果进一步的提示词工程无法改善摘要，可以从 Phoenix 导出数据集，使用 [OpenAI 微调 API](https://platform.openai.com/docs/guides/fine-tuning/create-a-fine-tuned-model) 训练满足需求的专用模型。